# Axis 2: Topic Discovery & Labelling Methods – Overview

**Responsible:** Member C (Topic & Labeling Lead)  
**Objective:** Compare different issue-discovery and labelling methods on customer-support tickets.  
**Sub-notebooks:**
- `03_Topic_and_Insights/03a_Topic_Modeling.ipynb` – Unsupervised topic discovery (LDA / NMF / BERTopic optional)
- `03_Topic_and_Insights/03b_Sentiment_Labeling.ipynb` – Semi-automated labelling (sentiment + issue type)
- `03_Topic_and_Insights/03c_Comparison_Analysis.ipynb` – Evaluation and comparison of both approaches


## Task 2c – Hypothesis

> **H1 (Topic Modeling):** Unsupervised topic models (LDA / NMF) will surface at least 4 coherent, human-interpretable ticket themes (e.g. login issues, payment errors, feature requests, account management) with a topic-coherence score (C_v) > 0.45.

> **H2 (Sentiment Labeling):** A rule-based issue-type classifier will cover ≥ 80 % of tickets when combined with VADER sentiment scoring, producing actionable labels without any training data.

> **H3 (Comparison):** The rule-based labelling pipeline will achieve higher *label coverage* on short tickets (< 20 tokens) while topic models will outperform on longer, noisier tickets in terms of *coherence* and *diversity of discovered themes*.


## 1. Environment Setup

In [1]:
# ── Standard library ──────────────────────────────────────────────────────
import os, json, warnings
warnings.filterwarnings('ignore')

# ── Third-party ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Output directory ──────────────────────────────────────────────────────
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')


Results will be saved to: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results


## 2. Virtual Dataset Generator

Run this cell to generate a synthetic customer-support ticket dataset (`virtual_tickets.csv`) that all three sub-notebooks can share.

In [2]:
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

# ── Topic templates ───────────────────────────────────────────────────────
TEMPLATES = {
    'login': [
        'I cannot log in to my account. The password reset email never arrived.',
        'Two-factor authentication code is not working on my phone.',
        'My account is locked after multiple failed login attempts.',
        'Login page keeps redirecting me to an error 403 page.',
        'Forgot my password and the reset link expired before I could use it.',
        'Single sign-on with Google stopped working this morning.',
        'Session keeps expiring after only 5 minutes, forcing constant re-login.',
    ],
    'bug': [
        'The app crashes every time I try to upload a file larger than 10 MB.',
        'Error 500 is displayed whenever I submit the checkout form.',
        'The dashboard chart is blank – no data is shown despite records existing.',
        'Notifications are duplicated; I receive the same alert three times.',
        'Sorting the table by date column causes the page to freeze.',
        'The export to CSV button downloads an empty file.',
        'Dark mode toggle reverts to light mode after every page refresh.',
    ],
    'feature': [
        'It would be great if we could export reports directly to PDF.',
        'Please add support for bulk-deleting records from the list view.',
        'Can you implement a calendar view for upcoming deadlines?',
        'We need an API endpoint for programmatic data ingestion.',
        'Would love a keyboard shortcut to quickly create a new ticket.',
        'Please allow customisable notification frequency per user.',
        'A mobile app for iOS would significantly improve our workflow.',
    ],
    'payment': [
        'My credit card was charged twice for the same subscription renewal.',
        'Invoice from last month is missing from my billing history.',
        'PayPal checkout fails at the confirmation step with a timeout error.',
        'I cancelled my subscription but was still billed this month.',
        'Need a refund for the accidental purchase made last week.',
        'The promo code I applied is not reflected on the final invoice.',
        'VAT is not applied correctly for EU customers in the billing portal.',
    ],
    'account': [
        'I need to update the email address associated with my account.',
        'How do I transfer ownership of my workspace to a colleague?',
        'My account was deactivated without any prior notification.',
        'Please help me delete my account and all personal data under GDPR.',
        'I cannot change my display name – the save button does nothing.',
        'My profile picture is not updating even after multiple uploads.',
        'Need to add a secondary admin user to our organisation account.',
    ],
}

ISSUE_TYPE_MAP = {
    'login': 'Account', 'bug': 'Bug', 'feature': 'Feature',
    'payment': 'Bug', 'account': 'Account',
}

SENTIMENT_MAP = {
    'login':   ['negative', 'negative', 'neutral', 'negative', 'neutral', 'negative', 'negative'],
    'bug':     ['negative', 'negative', 'neutral',  'negative', 'negative', 'neutral', 'neutral'],
    'feature': ['positive', 'neutral',  'positive', 'neutral',  'positive', 'positive', 'positive'],
    'payment': ['negative', 'neutral',  'negative', 'negative', 'neutral',  'neutral',  'negative'],
    'account': ['neutral',  'neutral',  'negative', 'neutral',  'neutral',  'neutral',  'neutral'],
}

N_TICKETS   = 200          # ← adjust to generate more data
START_DATE  = datetime(2024, 1, 1)

rows = []
topics = list(TEMPLATES.keys())
for i in range(N_TICKETS):
    topic   = random.choice(topics)
    idx     = random.randint(0, len(TEMPLATES[topic]) - 1)
    text    = TEMPLATES[topic][idx]
    date    = START_DATE + timedelta(days=random.randint(0, 364))
    rows.append({
        'ticket_id':        f'TKT-{i+1:04d}',
        'text':             text,
        'true_topic':       topic,
        'true_issue_type':  ISSUE_TYPE_MAP[topic],
        'true_sentiment':   SENTIMENT_MAP[topic][idx],
        'created_at':       date.strftime('%Y-%m-%d'),
    })

df = pd.DataFrame(rows)
out_path = os.path.join(RESULTS_DIR, 'virtual_tickets.csv')
df.to_csv(out_path, index=False)
print(f'Virtual dataset saved → {out_path}')
print(f'Shape: {df.shape}')
df.head(5)


Virtual dataset saved → /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/virtual_tickets.csv
Shape: (200, 6)


,ticket_id,text,true_topic,true_issue_type,true_sentiment,created_at
0,TKT-0001,I cannot log in to my account. The password re...,login,Account,negative,2024-05-20
1,TKT-0002,Error 500 is displayed whenever I submit the c...,bug,Bug,negative,2024-03-12
2,TKT-0003,Single sign-on with Google stopped working thi...,login,Account,negative,2024-10-06
3,TKT-0004,Forgot my password and the reset link expired ...,login,Account,neutral,2024-08-04
4,TKT-0005,I cannot log in to my account. The password re...,login,Account,negative,2024-02-17


## 3. Dataset Quick-look

In [3]:
print('Topic distribution:')
print(df['true_topic'].value_counts())
print('\nIssue-type distribution:')
print(df['true_issue_type'].value_counts())
print('\nSentiment distribution:')
print(df['true_sentiment'].value_counts())


Topic distribution:
true_topic
login      43
account    43
bug        41
payment    39
feature    34
Name: count, dtype: int64

Issue-type distribution:
true_issue_type
Account    86
Bug        80
Feature    34
Name: count, dtype: int64

Sentiment distribution:
true_sentiment
neutral     95
negative    80
positive    25
Name: count, dtype: int64


## 4. Next Steps

Open and run the sub-notebooks **in order**:

1. `03_Topic_and_Insights/03a_Topic_Modeling.ipynb`
2. `03_Topic_and_Insights/03b_Sentiment_Labeling.ipynb`
3. `03_Topic_and_Insights/03c_Comparison_Analysis.ipynb`

All outputs will be written to the `results/` folder.
